# Exact-OT operator and the final Gaussian-W2 map (Table 1, `tab:rules` FID)

Renders the exact (one-to-one) OT operator at n=1,000 for Table 1, computes FID by rule for `tab:rules`, and produces the final Gaussian-W2 map.

**Needs:** DTD, the CNT checkpoint, `matching_ablation.ipynb`'s cached reference/features.

**Maintained runnable path:** `experiments/cnt_1000.py` renders the same exact-OT operator; `experiments/rules_200.py` covers `tab:rules`'s FID-by-rule step this notebook also touches. Kept for provenance.

# Finish run — exact operator at n=1000 (+ FID by rule)
Only what's left, optimized:
- **Setup ~5 min**: DTD via one zip (created on first run, reused after), minimal pip set, detectron2 only if the import actually fails, all reference features loaded from the Drive caches the earlier runs already wrote.
- **All scoring batched** (64 images per forward pass) instead of one image per call — this is what made the last run 10× slower than needed.
- **Everything resumable**: every block checks Drive before computing; a disconnect costs the current block only. Use Runtime → "Run all"; safe to rerun after reconnect.

In [ ]:
# ============================================================
# 0. FAST SETUP
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'; os.makedirs(OUT,exist_ok=True)
FIGS=f'{OUT}/figs'; os.makedirs(FIGS,exist_ok=True); TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
NUM={}; note=lambda k,v:(NUM.__setitem__(k,v),print(f'  {k} = {v}'))[0]

t0=time.time()
# --- DTD via single zip (created once, then reused forever) ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    if not os.path.exists(ZIP):
        print('first run: building dtd zip on Drive (one-time, ~4 min)')
        os.system(f'cd "{BASE}/dtd" && zip -qr "{ZIP}" images')
    os.system(f'unzip -qo "{ZIP}" -d /content/_dtd && mv /content/_dtd/images {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')

# --- minimal pip (torch/scipy/sklearn preinstalled on Colab) ---
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
print(f'model ready {time.time()-t0:.0f}s')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def _dec(d,it,L,H,W,Cc,layout='a'):
    it=it.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_"+layout],{s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def interp_exact(d,eta,seed,n_exact=1024):
    L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t'); src,tgt=sf[i_s],tf[i_t]
    g=torch.Generator(device=device).manual_seed(seed+11)
    js=torch.randperm(NSUB,generator=g,device=device)[:n_exact]; is_=torch.randperm(NSUB,generator=g,device=device)[:n_exact]
    r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
    tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
    ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,Cc)

# --- observers, BATCHED ---
import timm
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)

ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def fid(X,Y):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False)
    cm,_=linalg.sqrtm(Sx@Sy,disp=False); cm=cm.real if np.iscomplexobj(cm) else cm
    return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
print(f'setup done in {time.time()-t0:.0f}s')

In [ ]:
import os, glob, time
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy; ZIP=f'{BASE}/dtd_images.zip'; LOCAL='/content/dtd_images'
print('zip on drive:', os.path.exists(ZIP), os.path.getsize(ZIP)/1e6 if os.path.exists(ZIP) else 0, 'MB')
print('local imgs:', len(glob.glob(f'{LOCAL}/*/*.jpg')))

t0=time.time()
if len(glob.glob(f'{LOCAL}/*/*.jpg')) < 5000:
    if os.path.exists(ZIP) and os.path.getsize(ZIP) > 500e6:      # a complete zip is ~600 MB
        os.system(f'unzip -qo "{ZIP}" -d /content/_dtd && rm -rf {LOCAL} && mv /content/_dtd/images {LOCAL}')
    else:
        if os.path.exists(ZIP): os.remove(ZIP)                     # partial zip from the interrupted run
        # copy Drive->local as a stream (tar over the mount reads sequentially: much faster than per-file cp),
        # then zip LOCALLY, then move one file to Drive
        os.system(f'cd "{BASE}/dtd" && tar cf - images | (cd /content && tar xf -)')
        os.system(f'mv /content/images {LOCAL}')
        os.system(f'cd /content && zip -qr /content/dtd_images.zip dtd_images && mv /content/dtd_images.zip "{ZIP}"')
print(len(glob.glob(f'{LOCAL}/*/*.jpg')), 'images,', f'{time.time()-t0:.0f}s')

In [ ]:
import os, glob, time
t0=time.time()
os.system('wget -q https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz -O /content/dtd.tar.gz')
os.system('cd /content && tar xzf dtd.tar.gz dtd/images && rm -rf /content/dtd_images && mv dtd/images /content/dtd_images')
print(len(glob.glob('/content/dtd_images/*/*.jpg')), 'images in', f'{time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# 1. FID BY MATCHING RULE — GPU eigvalsh instead of scipy sqrtm (the sqrtm bootstrap was O(hours); this is ~2 min)
#    trace(sqrtm(Sx·Sy)) = Σ √λ of the symmetric product  Sx^{1/2} Sy Sx^{1/2}
# ============================================================
RULES=['ot','ot_sharp','exact','nn','gauss','uniform','random','pixel_linear','texton_linear']
feats=dict(np.load(f'{OUT}/rules_feats.npz',allow_pickle=True)); dfr=pd.read_csv(f'{OUT}/rules_n200.csv'); ks=sorted(dfr.k.unique())
Xr={r:torch.tensor(np.stack([feats[f'{k}_{r}_inc'] for k in ks]),device=device,dtype=torch.float64) for r in RULES}
R =torch.tensor(ref_inc,device=device,dtype=torch.float64)

def cov_t(X):
    Xc=X-X.mean(0,keepdim=True); return Xc.T@Xc/(X.shape[0]-1)
mR,SR=R.mean(0),cov_t(R)
eR,vR=torch.linalg.eigh(SR); SRh=(vR*eR.clamp_min(0).sqrt())@vR.T   # SR^{1/2}, once

def fid_gpu(X):
    m,S=X.mean(0),cov_t(X)
    lam=torch.linalg.eigvalsh(SRh@S@SRh).clamp_min(0)
    return float(((m-mR)**2).sum()+torch.trace(S)+torch.trace(SR)-2*lam.sqrt().sum())

t0=time.time(); B=100; rng=np.random.RandomState(0)
FIDpt={r:fid_gpu(Xr[r]) for r in RULES}; boot={r:[] for r in RULES}
for b in range(B):
    idx=torch.tensor(rng.choice(len(ks),len(ks),replace=True),device=device)
    for r in RULES: boot[r].append(fid_gpu(Xr[r][idx]))
    if b%20==0: print(f'boot {b}/{B}  {time.time()-t0:.0f}s')
draws={r:np.array(boot[r]) for r in RULES}
print(f"\n{'rule':14}{'FID':>8}{'95% CI':>18}{'P(FID<ot)':>11}")
for r in sorted(RULES,key=lambda r:FIDpt[r]):
    lo,hi=np.percentile(draws[r],[2.5,97.5]); p=float((draws[r]<draws['ot']).mean()) if r!='ot' else float('nan')
    print(f"{r:14}{FIDpt[r]:8.1f}   [{lo:6.1f},{hi:6.1f}]{p:11.2f}")
json.dump({r:[FIDpt[r],*map(float,np.percentile(draws[r],[2.5,97.5]))] for r in RULES},open(f'{OUT}/fid_by_rule.json','w'),indent=1)

In [ ]:
# ============================================================
# 2. EXACT: subsample check (5 pairs) + generate midpoint & endpoint for all 1000 (resumable, ~0.6 s/pair)
# ============================================================
import lpips as lpips_lib
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
CHK=f'{OUT}/exact_subsample_check.json'
if not os.path.exists(CHK):
    chk=[]
    for p_ in MAN_ALL[:5]:
        k=int(p_['k']); A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); Bp=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(Bp))
        m1=to_pil(interp_exact(d,0.5,k,1024)); m2=to_pil(interp_exact(d,0.5,k,2048)); chk.append(lpips_d(m1,m2))
    json.dump(chk,open(CHK,'w'))
chk=json.load(open(CHK)); note('ExactSubsampleCheckLPIPS',round(float(np.median(chk)),3))

EXD=f'{SCALE}/exact'; os.makedirs(EXD,exist_ok=True)
todo=[int(p_['k']) for p_ in MAN_ALL if not (os.path.exists(f'{EXD}/{int(p_["k"]):04d}.png') and os.path.exists(f'{EXD}/{int(p_["k"]):04d}_end.png'))]
print(f'{len(todo)} pairs to generate'); t0=time.time()
for n,k in enumerate(todo):
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); Bp=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(Bp))
    to_pil(interp_exact(d,0.5,k)).save(f'{EXD}/{k:04d}.png'); to_pil(interp_exact(d,1.0,k)).save(f'{EXD}/{k:04d}_end.png')
    if n%50==0 and n: el=time.time()-t0; print(f'{n}/{len(todo)}  {el/n:.2f}s/pair  ETA {(len(todo)-n)*el/n/60:.0f} min')
print('exact generation complete')

In [ ]:
# ============================================================
# 3. RECON ANCHOR FEATURES for all 1000 pairs (batched; cached once, reused by any later scoring)
# ============================================================
ANC=f'{OUT}/anchor_dino.npz'
if os.path.exists(ANC): anc=dict(np.load(ANC))
else:
    anc={}; batchA=[]; batchB=[]; kk=[]
    t0=time.time()
    for i,p_ in enumerate(MAN_ALL):
        k=int(p_['k'])
        batchA.append(to_pil(make_recon(Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'))))
        batchB.append(to_pil(make_recon(Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')))); kk.append(k)
        if len(kk)==64 or i==len(MAN_ALL)-1:
            fA=dino_feats(batchA); fB=dino_feats(batchB)
            for j,k2 in enumerate(kk): anc[f'{k2}_A']=fA[j]; anc[f'{k2}_B']=fB[j]
            batchA,batchB,kk=[],[],[]; print(f'{i+1}/1000  {time.time()-t0:.0f}s'); np.savez(ANC,**anc)
    np.savez(ANC,**anc)
print('anchors:',len(anc)//2,'pairs')

In [ ]:
# ============================================================
# 4. SCORE all methods from their PNG dirs (batched) → final table with FID/KID/realism/placement + paired stats
# ============================================================
METHODS={m:p for m,p in {'exact':f'{SCALE}/exact','ot':f'{SCALE}/ot','ot_sym':f'{SCALE}/ot_sym',
         'pixel_linear':f'{SCALE}/pixel_linear','texton_linear':f'{SCALE}/texton_linear','tm':f'{SCALE}/mid'}.items() if os.path.isdir(p)}
anc=dict(np.load(f'{OUT}/anchor_dino.npz'))
def method_feats(m,pdir):
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'
    if os.path.exists(fI): return np.load(fI),np.load(fD)
    fs=[f'{pdir}/{int(p_["k"]):04d}.png' for p_ in MAN_ALL]; fs=[f for f in fs if os.path.exists(f)]
    ims=[Image.open(f).convert('RGB') for f in fs]
    XI=inc_feats(ims); XD=dino_feats(ims); np.save(fI,XI); np.save(fD,XD); return XI,XD
rows=[]; XIs={}
for m,pdir in METHODS.items():
    t0=time.time(); XI,XD=method_feats(m,pdir); XIs[m]=XI
    ks=[int(p_['k']) for p_ in MAN_ALL if os.path.exists(f'{pdir}/{int(p_["k"]):04d}.png')]
    rI=realism(XI,MANI_INC); rD=realism(XD,MANI_DINO)
    plc=np.array([np.linalg.norm(XD[i]-anc[f'{k}_A'])/(np.linalg.norm(XD[i]-anc[f'{k}_A'])+np.linalg.norm(XD[i]-anc[f'{k}_B'])+1e-9) for i,k in enumerate(ks)])
    for i,k in enumerate(ks): rows.append(dict(k=k,method=m,real_inc=rI[i],real_dino=rD[i],place=plc[i]))
    print(f'{m}: {len(ks)} imgs scored in {time.time()-t0:.0f}s')
sc=pd.DataFrame(rows); sc.to_csv(f'{OUT}/exact_n1000.csv',index=False)
# endpoint gap for exact
EG=f'{OUT}/exact_endgap.npy'
if os.path.exists(EG): eg=np.load(EG)
else:
    ims=[Image.open(f'{SCALE}/exact/{int(p_["k"]):04d}_end.png').convert('RGB') for p_ in MAN_ALL]
    XE=dino_feats(ims)
    eg=np.array([np.linalg.norm(XE[i]-anc[f'{int(p_["k"])}_B'])/ (np.linalg.norm(anc[f'{int(p_["k"])}_A']-anc[f'{int(p_["k"])}_B'])+1e-9) for i,p_ in enumerate(MAN_ALL)])
    np.save(EG,eg)
note('EndpointGap_exact',round(float(np.median(eg)),3))

piv={c:sc.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
print(f"\n{'method':14}{'FID1000':>9}{'KID1000':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}   exact-vs: dRealInc [95%CI] p")
for m in METHODS:
    Fv=fid(XIs[m],ref_inc); Kv=kid(XIs[m],ref_inc)
    line=f"{m:14}{Fv:9.1f}{Kv[0]:9.4f}±{Kv[1]:.4f}{piv['real_inc'][m].median():9.3f}{piv['real_dino'][m].median():10.3f}{(piv['place'][m]-.5).abs().median():11.3f}"
    if m!='exact':
        dlt=(piv['real_inc']['exact']-piv['real_inc'][m]).dropna().values
        lo,hi=boot_ci(dlt); line+=f"   {np.median(dlt):+.3f} [{lo:+.3f},{hi:+.3f}] p={wilcoxon(dlt).pvalue:.2g}"
    print(line)
    note(f'FIDk_{m}',round(Fv,1)); note(f'KIDk_{m}',round(Kv[0],4))
    note(f'RealInc_{m}',round(float(piv['real_inc'][m].median()),3)); note(f'RealDino_{m}',round(float(piv['real_dino'][m].median()),3))
    note(f'Place_{m}',round(float(abs(piv['place'][m]-.5).median()),3))
json.dump(NUM,open(f'{OUT}/numbers_exact.json','w'),indent=1); print('\nsaved numbers_exact.json')

In [ ]:
# ============================================================
# OT vs GAUSSIAN-OT — FINAL LARGE-SCALE TEST
#
# Gaussian baseline:
#   Fit N(mu_A, Sigma_A), N(mu_B, Sigma_B) on the SAME
#   deterministic 4096-point style-map subsamples as OT.
#   Apply the closed-form Gaussian W2 optimal affine map:
#
#   T(x) = mu_B + A (x-mu_A)
#   A = Sa^{-1/2} (Sa^{1/2} Sb Sa^{1/2})^{1/2} Sa^{-1/2}
#
# Then use the same interpolation:
#   X_0.5 = 0.5 X + 0.5 T(X)
#
# IMPORTANT:
# - Reuses existing OT PNG/features: OT is NOT regenerated.
# - Gaussian images/features are cached and resumable.
# - No FID bootstrap (biased at finite n).
# - Reports FID, KID, Inception/DINO realism, midpoint placement,
#   paired bootstrap CIs + Wilcoxon tests.
# ============================================================

import os, time, gc, json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import wilcoxon

# ---------- SETTINGS ----------
N_TEST = min(1000, len(MAN_ALL))       # final paper: 1000
ETA = 0.5
GAUSS_DIR = f'{SCALE}/gaussian_ot'
os.makedirs(GAUSS_DIR, exist_ok=True)

pairs = MAN_ALL[:N_TEST]
pair_ks = [int(p['k']) for p in pairs]

# ---------- PSD matrix utilities ----------
@torch.no_grad()
def _sym(M):
    return 0.5 * (M + M.T)

@torch.no_grad()
def _psd_sqrt(M, floor):
    """Symmetric PSD square root."""
    e, V = torch.linalg.eigh(_sym(M))
    e = e.clamp_min(floor)
    return (V * e.sqrt().unsqueeze(0)) @ V.T

@torch.no_grad()
def _psd_invsqrt(M, floor):
    """Symmetric PSD inverse square root."""
    e, V = torch.linalg.eigh(_sym(M))
    e = e.clamp_min(floor)
    return (V * e.rsqrt().unsqueeze(0)) @ V.T


# ---------- Gaussian W2 affine interpolation ----------
@torch.no_grad()
def interp_gaussian_ot(d, eta=0.5, seed=0):
    """
    Gaussian W2 optimal affine map in CNT style-map space.

    Fits mean/covariance using the SAME deterministic NSUB=4096
    source/target samples used by the OT implementation, but applies
    the resulting affine transform to the full source style map.
    """
    L, Cc, H, W, sf, tf = _grids(d)
    N = sf.shape[0]

    # Same deterministic subsampling protocol as OT
    i_s = _sub(N, seed, 's')
    i_t = _sub(N, seed, 't')

    X = sf[i_s].float()
    Y = tf[i_t].float()

    mu_x = X.mean(0)
    mu_y = Y.mean(0)

    Xc = X - mu_x
    Yc = Y - mu_y

    Sa = (Xc.T @ Xc) / max(len(X) - 1, 1)
    Sb = (Yc.T @ Yc) / max(len(Y) - 1, 1)

    # Scale-aware ridge only for numerical stability.
    # Tiny relative to average variance.
    scale = 0.5 * (torch.trace(Sa) + torch.trace(Sb)) / Cc
    ridge = max(float(scale) * 1e-5, 1e-8)

    I = torch.eye(Cc, device=sf.device, dtype=torch.float32)
    Sa = _sym(Sa) + ridge * I
    Sb = _sym(Sb) + ridge * I

    # Gaussian 2-Wasserstein optimal transport map
    Sa_half = _psd_sqrt(Sa, ridge)
    Sa_ihalf = _psd_invsqrt(Sa, ridge)

    middle = _sym(Sa_half @ Sb @ Sa_half)
    middle_half = _psd_sqrt(middle, ridge)

    A = _sym(Sa_ihalf @ middle_half @ Sa_ihalf)

    # Apply affine Gaussian map to ALL source cells.
    # Chunking avoids a large temporary allocation and is fast on A100.
    Tfull = torch.empty_like(sf, dtype=torch.float32)
    CHUNK = 16384
    for s in range(0, len(sf), CHUNK):
        xx = sf[s:s+CHUNK].float()
        Tfull[s:s+CHUNK] = mu_y + (xx - mu_x) @ A.T

    Xeta = (1.0 - eta) * sf.float() + eta * Tfull
    return _dec(d, Xeta, L, H, W, Cc)


# ============================================================
# 1) GENERATE ONLY MISSING GAUSSIAN MIDPOINTS
# ============================================================
todo = [
    p for p in pairs
    if not os.path.exists(f'{GAUSS_DIR}/{int(p["k"]):04d}.png')
]

print(f'Gaussian outputs: {N_TEST-len(todo)}/{N_TEST} cached; '
      f'{len(todo)} to generate.')

t0 = time.time()

for n, p in enumerate(todo, 1):
    k = int(p['k'])

    A_img = Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB')
    B_img = Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

    d = build_d(cnt_load(A_img), cnt_load(B_img))
    out = interp_gaussian_ot(d, ETA, k)
    to_pil(out).save(f'{GAUSS_DIR}/{k:04d}.png')

    del d, out
    if n % 50 == 0 or n == len(todo):
        elapsed = time.time() - t0
        rate = elapsed / n
        eta_min = (len(todo) - n) * rate / 60
        print(f'{n:4d}/{len(todo)}  '
              f'{rate:.2f}s/pair  ETA {eta_min:.1f} min')

    if n % 100 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print(f'Gaussian generation finished in {(time.time()-t0)/60:.1f} min')


# ============================================================
# 2) FEATURES — CACHE GAUSSIAN, REUSE EXISTING OT
# ============================================================
GI_path = f'{OUT}/feat_gaussian_ot_n{N_TEST}_inc.npy'
GD_path = f'{OUT}/feat_gaussian_ot_n{N_TEST}_dino.npy'

if os.path.exists(GI_path) and os.path.exists(GD_path):
    GI = np.load(GI_path)
    GD = np.load(GD_path)
    print('Loaded cached Gaussian observer features.')
else:
    print('Scoring Gaussian outputs...')
    ims = [
        Image.open(f'{GAUSS_DIR}/{k:04d}.png').convert('RGB')
        for k in pair_ks
    ]

    tscore = time.time()
    GI = inc_feats(ims, bs=64)
    GD = dino_feats(ims, bs=64)

    np.save(GI_path, GI)
    np.save(GD_path, GD)
    del ims
    print(f'Gaussian scoring: {(time.time()-tscore)/60:.1f} min')


# OT features already produced by your previous final scoring cell
OI_path = f'{OUT}/feat_ot_inc.npy'
OD_path = f'{OUT}/feat_ot_dino.npy'

if os.path.exists(OI_path) and os.path.exists(OD_path):
    OI = np.load(OI_path)[:N_TEST]
    OD = np.load(OD_path)[:N_TEST]
    print('Loaded cached OT observer features.')
else:
    print('OT feature cache missing — scoring existing OT PNGs only.')
    ot_ims = [
        Image.open(f'{SCALE}/ot/{k:04d}.png').convert('RGB')
        for k in pair_ks
    ]
    OI = inc_feats(ot_ims, bs=64)
    OD = dino_feats(ot_ims, bs=64)
    del ot_ims


# ============================================================
# 3) REALISM + PLACEMENT
# ============================================================
rI_ot = realism(OI, MANI_INC)
rI_g  = realism(GI, MANI_INC)

rD_ot = realism(OD, MANI_DINO)
rD_g  = realism(GD, MANI_DINO)

anc = dict(np.load(f'{OUT}/anchor_dino.npz'))

def placement(XD):
    vals = []
    for i, k in enumerate(pair_ks):
        a = anc[f'{k}_A']
        b = anc[f'{k}_B']
        da = np.linalg.norm(XD[i] - a)
        db = np.linalg.norm(XD[i] - b)
        vals.append(da / (da + db + 1e-9))
    return np.asarray(vals)

pl_ot = placement(OD)
pl_g  = placement(GD)

bal_ot = np.abs(pl_ot - 0.5)
bal_g  = np.abs(pl_g  - 0.5)


# ============================================================
# 4) SET-LEVEL METRICS
# ============================================================
FID_ot = fid(OI, ref_inc)
FID_g  = fid(GI, ref_inc)

KID_ot, KID_ot_sd = kid(OI, ref_inc, n_rep=50)
KID_g,  KID_g_sd  = kid(GI, ref_inc, n_rep=50)


# ============================================================
# 5) PAIRED STATISTICS
# ============================================================
def paired_report(name, g, o, higher_better=True):
    """
    delta = Gaussian - OT.
    Positive is good for Gaussian when higher_better=True.
    """
    d = np.asarray(g) - np.asarray(o)
    med = float(np.median(d))
    lo, hi = boot_ci(d, n=5000, seed=17)

    # avoid Wilcoxon crashing only in the pathological all-zero case
    try:
        p = float(wilcoxon(d).pvalue)
    except ValueError:
        p = 1.0

    if higher_better:
        win = float(np.mean(d > 0))
        interpretation = "Gaussian wins" if med > 0 else "OT wins"
    else:
        win = float(np.mean(d < 0))
        interpretation = "Gaussian wins" if med < 0 else "OT wins"

    print(
        f'{name:20s} Δ(G-OT)={med:+.4f}  '
        f'95%CI [{lo:+.4f},{hi:+.4f}]  '
        f'p={p:.3g}  Gaussian-win={100*win:.1f}%  '
        f'→ {interpretation}'
    )
    return dict(delta_median=med, ci=[float(lo),float(hi)],
                p=p, gaussian_win_rate=win)


# ============================================================
# 6) FINAL TABLE
# ============================================================
print('\n' + '='*90)
print(f'OT vs Gaussian W2 affine map — n={N_TEST}')
print('='*90)

print(
    f'{"Method":12s} '
    f'{"FID↓":>9s} {"KID↓":>16s} '
    f'{"RealInc↑":>11s} {"RealDINO↑":>12s} '
    f'{"|place-.5|↓":>13s}'
)

print(
    f'{"OT":12s} '
    f'{FID_ot:9.1f} '
    f'{KID_ot:8.4f}±{KID_ot_sd:.4f} '
    f'{np.median(rI_ot):11.3f} '
    f'{np.median(rD_ot):12.3f} '
    f'{np.median(bal_ot):13.3f}'
)

print(
    f'{"Gaussian":12s} '
    f'{FID_g:9.1f} '
    f'{KID_g:8.4f}±{KID_g_sd:.4f} '
    f'{np.median(rI_g):11.3f} '
    f'{np.median(rD_g):12.3f} '
    f'{np.median(bal_g):13.3f}'
)

print('\nPaired per-image comparisons:')
stats = {}
stats['realism_inception'] = paired_report(
    'Inception realism', rI_g, rI_ot, higher_better=True
)
stats['realism_dino'] = paired_report(
    'DINO realism', rD_g, rD_ot, higher_better=True
)
stats['placement_error'] = paired_report(
    'Placement error', bal_g, bal_ot, higher_better=False
)

print('\nSet-level differences:')
print(f'  ΔFID  (Gaussian - OT) = {FID_g-FID_ot:+.2f}  '
      f'({"OT better" if FID_g > FID_ot else "Gaussian better"})')
print(f'  ΔKID  (Gaussian - OT) = {KID_g-KID_ot:+.5f}  '
      f'({"OT better" if KID_g > KID_ot else "Gaussian better"})')

# Save clean machine-readable result
result = {
    'n': N_TEST,
    'definition': 'Gaussian W2 optimal affine map fitted on deterministic NSUB style-map samples',
    'ot': {
        'fid': FID_ot,
        'kid': KID_ot,
        'kid_sd': KID_ot_sd,
        'realism_inc_median': float(np.median(rI_ot)),
        'realism_dino_median': float(np.median(rD_ot)),
        'placement_error_median': float(np.median(bal_ot)),
    },
    'gaussian': {
        'fid': FID_g,
        'kid': KID_g,
        'kid_sd': KID_g_sd,
        'realism_inc_median': float(np.median(rI_g)),
        'realism_dino_median': float(np.median(rD_g)),
        'placement_error_median': float(np.median(bal_g)),
    },
    'paired': stats,
}

with open(f'{OUT}/ot_vs_gaussian_n{N_TEST}.json', 'w') as f:
    json.dump(result, f, indent=2)

print(f'\nSaved → {OUT}/ot_vs_gaussian_n{N_TEST}.json')

In [ ]:
# how ill-conditioned is A, and what do the images look like?
import numpy as np, torch
from PIL import Image
import matplotlib.pyplot as plt
p_=MAN_ALL[0]; k=int(p_['k'])
A_img=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B_img=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
d=build_d(cnt_load(A_img),cnt_load(B_img)); L,Cc,H,W,sf,tf=_grids(d)
i_s=_sub(sf.shape[0],k,'s'); i_t=_sub(tf.shape[0],k,'t'); X,Y=sf[i_s].float(),tf[i_t].float()
Sa=torch.cov(X.T); Sb=torch.cov(Y.T)
for ridge in [float(0.5*(torch.trace(Sa)+torch.trace(Sb))/Cc)*1e-5, 1e-4, 1e-3]:
    I=torch.eye(Cc,device=device); Sh=_psd_sqrt(Sa+ridge*I,ridge); Sih=_psd_invsqrt(Sa+ridge*I,ridge)
    Amap=_sym(Sih@_psd_sqrt(_sym(Sh@(Sb+ridge*I)@Sh),ridge)@Sih)
    e=torch.linalg.eigvalsh(Amap)
    # norm of transported FULL cloud vs transported SAMPLE
    Tfull=(sf.float()-X.mean(0))@Amap.T+Y.mean(0); Tsamp=(X-X.mean(0))@Amap.T+Y.mean(0)
    print(f'ridge={ridge:.2e}  eig(A) [{e.min():.3f},{e.max():.1f}]  |T| samp {Tsamp.norm(dim=1).median():.2f} / full {Tfull.norm(dim=1).median():.2f} max {Tfull.norm(dim=1).max():.1f}  vs |Y| {Y.norm(dim=1).median():.2f}')
fig,ax=plt.subplots(1,3,figsize=(10,3.5))
ax[0].imshow(Image.open(f'{GAUSS_DIR}/{k:04d}.png')); ax[0].set_title('gaussian (this cell)')
ax[1].imshow(Image.open(f'{SCALE}/ot/{k:04d}.png')); ax[1].set_title('ot')
ax[2].imshow(to_pil(interp_rule(d,0.5,'gauss',seed=k)) if 'interp_rule' in dir() else A_img); ax[2].set_title('gauss (ablation impl)')
for a in ax: a.axis('off')
plt.show()

In [ ]:
# ============================================================
# GAUSSIAN W2 BASELINE, corrected: ridge 1e-4, transport the NSUB samples, NN-extend like every other rule
# ============================================================
import shutil
# clear the broken run (images + features), keep the dir
for k in pair_ks:
    f=f'{GAUSS_DIR}/{k:04d}.png'
    if os.path.exists(f): os.remove(f)
for f in [GI_path, GD_path]:
    if os.path.exists(f): os.remove(f)

RIDGE=1e-4
@torch.no_grad()
def interp_gaussian_ot(d, eta=0.5, seed=0):
    L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]
    i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    X,Y=sf[i_s].float(),tf[i_t].float()
    mu_x,mu_y=X.mean(0),Y.mean(0); Xc,Yc=X-mu_x,Y-mu_y
    I=torch.eye(Cc,device=sf.device)
    Sa=_sym(Xc.T@Xc/(len(X)-1))+RIDGE*I; Sb=_sym(Yc.T@Yc/(len(Y)-1))+RIDGE*I
    Sh=_psd_sqrt(Sa,RIDGE); Sih=_psd_invsqrt(Sa,RIDGE)
    Amap=_sym(Sih@_psd_sqrt(_sym(Sh@Sb@Sh),RIDGE)@Sih)
    tt=mu_y+Xc@Amap.T                       # transported SAMPLES only
    nn=torch.cdist(sf,sf[i_s]).argmin(1)    # same extension as ot / exact / nn / uniform / random
    return _dec(d,(1-eta)*sf.float()+eta*tt[nn],L,H,W,Cc)

# sanity on one pair before the loop
k=pair_ks[0]
d=build_d(cnt_load(Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB')),cnt_load(Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')))
plt.imshow(to_pil(interp_gaussian_ot(d,ETA,k))); plt.axis('off'); plt.title('gaussian, corrected'); plt.show()

In [ ]:
# ============================================================
# CORRECTED GAUSSIAN W2 — REGENERATE + SCORE
# IMPORTANT: run this AFTER all previous cells.
# This intentionally overwrites the old interp_gaussian_ot.
# ============================================================

import os, time, gc, json
import numpy as np
import torch
from PIL import Image
from scipy.stats import wilcoxon

RIDGE = 1e-4
N_TEST = min(1000, len(MAN_ALL))
ETA = 0.5

pairs = MAN_ALL[:N_TEST]
pair_ks = [int(p["k"]) for p in pairs]

GAUSS_DIR = f'{SCALE}/gaussian_ot_corrected'
os.makedirs(GAUSS_DIR, exist_ok=True)

GI_path = f'{OUT}/feat_gaussian_ot_corrected_n{N_TEST}_inc.npy'
GD_path = f'{OUT}/feat_gaussian_ot_corrected_n{N_TEST}_dino.npy'


# ------------------------------------------------------------
# Correct Gaussian W2 transport
# ------------------------------------------------------------
@torch.no_grad()
def interp_gaussian_ot(d, eta=0.5, seed=0):
    L, Cc, H, W, sf, tf = _grids(d)
    N = sf.shape[0]

    # exact same deterministic samples used by OT
    i_s = _sub(N, seed, 's')
    i_t = _sub(N, seed, 't')

    X = sf[i_s].float()
    Y = tf[i_t].float()

    mu_x = X.mean(0)
    mu_y = Y.mean(0)

    Xc = X - mu_x
    Yc = Y - mu_y

    I = torch.eye(Cc, device=sf.device, dtype=torch.float32)

    Sa = _sym(Xc.T @ Xc / (len(X)-1)) + RIDGE * I
    Sb = _sym(Yc.T @ Yc / (len(Y)-1)) + RIDGE * I

    Sa_half  = _psd_sqrt(Sa, RIDGE)
    Sa_ihalf = _psd_invsqrt(Sa, RIDGE)

    middle = _sym(Sa_half @ Sb @ Sa_half)
    middle_half = _psd_sqrt(middle, RIDGE)

    Amap = _sym(Sa_ihalf @ middle_half @ Sa_ihalf)

    # Transport ONLY the same 4096 sampled source cells.
    transported_samples = mu_y + Xc @ Amap.T

    # Same NN extension used by the other matching-rule baselines.
    nn = torch.cdist(sf, sf[i_s]).argmin(1)
    transported_full = transported_samples[nn]

    Xeta = (
        (1.0 - eta) * sf.float()
        + eta * transported_full
    )

    return _dec(d, Xeta, L, H, W, Cc)


# ------------------------------------------------------------
# Quick sanity BEFORE wasting 4 min
# ------------------------------------------------------------
k = pair_ks[0]

A0 = Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB')
B0 = Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

d0 = build_d(cnt_load(A0), cnt_load(B0))

test_out = to_pil(interp_gaussian_ot(d0, ETA, k))

display(test_out)

# numerical check
L,Cc,H,W,sf,tf = _grids(d0)
i_s = _sub(sf.shape[0],k,'s')
i_t = _sub(tf.shape[0],k,'t')

X = sf[i_s].float()
Y = tf[i_t].float()

mx,my = X.mean(0),Y.mean(0)
Xc,Yc = X-mx,Y-my

I = torch.eye(Cc,device=device)

Sa = _sym(Xc.T@Xc/(len(X)-1)) + RIDGE*I
Sb = _sym(Yc.T@Yc/(len(Y)-1)) + RIDGE*I

Sh  = _psd_sqrt(Sa,RIDGE)
Sih = _psd_invsqrt(Sa,RIDGE)

Amap = _sym(
    Sih
    @ _psd_sqrt(_sym(Sh@Sb@Sh),RIDGE)
    @ Sih
)

ee = torch.linalg.eigvalsh(Amap)

Ts = mx*0 + my + Xc@Amap.T

print(
    f"sanity: eig(A)=[{ee.min():.3f}, {ee.max():.1f}], "
    f"median |T(X)|={Ts.norm(dim=1).median():.2f}, "
    f"median |Y|={Y.norm(dim=1).median():.2f}"
)

del d0, test_out, X, Y, Ts
gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# CLEAR ONLY corrected-Gaussian files, so this run is guaranteed
# fresh.
# ------------------------------------------------------------
for k in pair_ks:
    fp = f'{GAUSS_DIR}/{k:04d}.png'
    if os.path.exists(fp):
        os.remove(fp)

for fp in [GI_path, GD_path]:
    if os.path.exists(fp):
        os.remove(fp)


# ------------------------------------------------------------
# Generate 1000 corrected Gaussian midpoints
# ------------------------------------------------------------
print(f'\nGenerating corrected Gaussian W2 outputs: {N_TEST}')

t0 = time.time()

for n,p in enumerate(pairs,1):

    k = int(p["k"])

    A_img = Image.open(
        f'{SCALE}/A/{k:04d}.png'
    ).convert('RGB')

    B_img = Image.open(
        f'{SCALE}/B/{k:04d}.png'
    ).convert('RGB')

    d = build_d(
        cnt_load(A_img),
        cnt_load(B_img)
    )

    out = interp_gaussian_ot(
        d,
        eta=ETA,
        seed=k
    )

    to_pil(out).save(
        f'{GAUSS_DIR}/{k:04d}.png'
    )

    del d,out

    if n%50==0:
        elapsed=time.time()-t0
        rate=elapsed/n
        remain=(N_TEST-n)*rate/60

        print(
            f'{n:4d}/{N_TEST}  '
            f'{rate:.2f}s/pair  '
            f'ETA {remain:.1f} min'
        )

    if n%100==0:
        gc.collect()
        torch.cuda.empty_cache()

print(
    f'Generation finished in '
    f'{(time.time()-t0)/60:.1f} min'
)


# ------------------------------------------------------------
# Batched observer scoring
# ------------------------------------------------------------
print('\nScoring corrected Gaussian outputs...')

ims = [
    Image.open(
        f'{GAUSS_DIR}/{k:04d}.png'
    ).convert('RGB')
    for k in pair_ks
]

ts=time.time()

GI = inc_feats(ims,bs=64)
GD = dino_feats(ims,bs=64)

np.save(GI_path,GI)
np.save(GD_path,GD)

del ims
gc.collect()

print(
    f'scoring finished in '
    f'{(time.time()-ts)/60:.1f} min'
)


# ------------------------------------------------------------
# Load canonical OT features
# ------------------------------------------------------------
OI = np.load(f'{OUT}/feat_ot_inc.npy')[:N_TEST]
OD = np.load(f'{OUT}/feat_ot_dino.npy')[:N_TEST]


# ------------------------------------------------------------
# Realism
# ------------------------------------------------------------
rI_ot = realism(OI,MANI_INC)
rI_g  = realism(GI,MANI_INC)

rD_ot = realism(OD,MANI_DINO)
rD_g  = realism(GD,MANI_DINO)


# ------------------------------------------------------------
# Midpoint placement
# ------------------------------------------------------------
anc = dict(np.load(f'{OUT}/anchor_dino.npz'))

def placement(XD):
    q=[]
    for i,k in enumerate(pair_ks):

        a=anc[f'{k}_A']
        b=anc[f'{k}_B']

        da=np.linalg.norm(XD[i]-a)
        db=np.linalg.norm(XD[i]-b)

        q.append(
            da/(da+db+1e-9)
        )

    return np.asarray(q)

pl_ot=placement(OD)
pl_g =placement(GD)

bal_ot=np.abs(pl_ot-.5)
bal_g =np.abs(pl_g-.5)


# ------------------------------------------------------------
# Set-level metrics
# ------------------------------------------------------------
FID_ot=fid(OI,ref_inc)
FID_g =fid(GI,ref_inc)

KID_ot,KID_ot_sd=kid(
    OI,ref_inc,
    n_rep=50
)

KID_g,KID_g_sd=kid(
    GI,ref_inc,
    n_rep=50
)


# ------------------------------------------------------------
# Paired inference
# ------------------------------------------------------------
def compare(name,g,o,higher=True):

    delta=np.asarray(g)-np.asarray(o)

    med=float(np.median(delta))
    lo,hi=boot_ci(
        delta,
        n=5000,
        seed=17
    )

    try:
        p=float(
            wilcoxon(delta).pvalue
        )
    except:
        p=1.0

    if higher:
        win=np.mean(delta>0)
    else:
        win=np.mean(delta<0)

    print(
        f'{name:20s} '
        f'Δ(G-OT)={med:+.4f}  '
        f'95%CI [{lo:+.4f},{hi:+.4f}]  '
        f'p={p:.3g}  '
        f'Gaussian-win={100*win:.1f}%'
    )


# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------
print('\n'+'='*90)

print(
    f'CORRECTED Gaussian W2 vs OT — '
    f'n={N_TEST}, ridge={RIDGE}'
)

print('='*90)

print(
    f'{"Method":12s} '
    f'{"FID↓":>9s} '
    f'{"KID↓":>16s} '
    f'{"RealInc↑":>11s} '
    f'{"RealDINO↑":>12s} '
    f'{"|place-.5|↓":>13s}'
)

print(
    f'{"OT":12s} '
    f'{FID_ot:9.1f} '
    f'{KID_ot:8.4f}±{KID_ot_sd:.4f} '
    f'{np.median(rI_ot):11.3f} '
    f'{np.median(rD_ot):12.3f} '
    f'{np.median(bal_ot):13.3f}'
)

print(
    f'{"Gaussian":12s} '
    f'{FID_g:9.1f} '
    f'{KID_g:8.4f}±{KID_g_sd:.4f} '
    f'{np.median(rI_g):11.3f} '
    f'{np.median(rD_g):12.3f} '
    f'{np.median(bal_g):13.3f}'
)

print('\nPaired comparisons:')

compare(
    'Inception realism',
    rI_g,rI_ot,
    True
)

compare(
    'DINO realism',
    rD_g,rD_ot,
    True
)

compare(
    'Placement error',
    bal_g,bal_ot,
    False
)

print('\nSet-level differences:')

print(
    f'ΔFID (G-OT) = '
    f'{FID_g-FID_ot:+.2f}'
)

print(
    f'ΔKID (G-OT) = '
    f'{KID_g-KID_ot:+.5f}'
)

In [ ]:
# ============================================================
# QUALITATIVE FIGURE — Gaussian W2 ridge sweep vs OT
#
# Columns:
# A | Gaussian 1e-4 | Gaussian 3e-4 | Gaussian 1e-3 | OT | B
#
# - 6 deterministic, non-cherry-picked pairs
# - reuses existing OT images
# - generates only 18 Gaussian images
# - saves high-resolution PNG + PDF for the paper
# ============================================================

import os, gc, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

RIDGES = [1e-4, 3e-4, 1e-3]
ETA = 0.5
N_ROWS = 6

# ------------------------------------------------------------
# Deterministic pair selection:
# evenly spaced through the 1000-pair manifest.
# We do NOT inspect the results before choosing them.
# ------------------------------------------------------------
idxs = np.linspace(0, len(MAN_ALL)-1, N_ROWS, dtype=int)
vis_pairs = [MAN_ALL[i] for i in idxs]

print("Selected manifest indices:", idxs.tolist())
print("Pair IDs:", [int(p["k"]) for p in vis_pairs])


# ------------------------------------------------------------
# Gaussian W2 with explicit ridge argument
# ------------------------------------------------------------
@torch.no_grad()
def interp_gaussian_ridge(d, eta=0.5, seed=0, ridge=1e-4):

    L, Cc, H, W, sf, tf = _grids(d)
    N = sf.shape[0]

    # Same deterministic 4096 samples as OT
    i_s = _sub(N, seed, 's')
    i_t = _sub(N, seed, 't')

    X = sf[i_s].float()
    Y = tf[i_t].float()

    mu_x = X.mean(0)
    mu_y = Y.mean(0)

    Xc = X - mu_x
    Yc = Y - mu_y

    I = torch.eye(
        Cc,
        device=sf.device,
        dtype=torch.float32
    )

    Sa = _sym(
        Xc.T @ Xc / (len(X)-1)
    ) + ridge * I

    Sb = _sym(
        Yc.T @ Yc / (len(Y)-1)
    ) + ridge * I

    Sa_half = _psd_sqrt(
        Sa,
        ridge
    )

    Sa_ihalf = _psd_invsqrt(
        Sa,
        ridge
    )

    middle = _sym(
        Sa_half @ Sb @ Sa_half
    )

    middle_half = _psd_sqrt(
        middle,
        0.0
    )

    Amap = _sym(
        Sa_ihalf
        @ middle_half
        @ Sa_ihalf
    )

    # transport sampled cells
    transported_samples = (
        mu_y
        + Xc @ Amap.T
    )

    # NN extension exactly like the other matching rules
    nn = torch.cdist(
        sf,
        sf[i_s]
    ).argmin(1)

    transported_full = transported_samples[nn]

    midpoint = (
        (1.0 - eta) * sf.float()
        + eta * transported_full
    )

    return _dec(
        d,
        midpoint,
        L,H,W,Cc
    )


# ------------------------------------------------------------
# Generate the images
# ------------------------------------------------------------
rows = []

t0 = time.time()

for row_i, p in enumerate(vis_pairs):

    k = int(p["k"])

    A_img = Image.open(
        f'{SCALE}/A/{k:04d}.png'
    ).convert('RGB')

    B_img = Image.open(
        f'{SCALE}/B/{k:04d}.png'
    ).convert('RGB')

    # existing OT midpoint
    OT_img = Image.open(
        f'{SCALE}/ot/{k:04d}.png'
    ).convert('RGB')

    d = build_d(
        cnt_load(A_img),
        cnt_load(B_img)
    )

    gaussian_imgs = []

    for ridge in RIDGES:

        out = interp_gaussian_ridge(
            d,
            eta=ETA,
            seed=k,
            ridge=ridge
        )

        gaussian_imgs.append(
            to_pil(out)
        )

        del out

    rows.append(
        [A_img]
        + gaussian_imgs
        + [OT_img, B_img]
    )

    del d
    gc.collect()

    print(
        f'pair {row_i+1}/{N_ROWS} '
        f'(k={k}) done'
    )

print(
    f'Generated {N_ROWS*len(RIDGES)} Gaussian '
    f'midpoints in {time.time()-t0:.1f}s'
)


# ------------------------------------------------------------
# Publication-style figure
# ------------------------------------------------------------
headers = [
    r'$A$',
    r'Gaussian $W_2$' + '\n' + r'$\lambda=10^{-4}$',
    r'Gaussian $W_2$' + '\n' + r'$\lambda=3\times10^{-4}$',
    r'Gaussian $W_2$' + '\n' + r'$\lambda=10^{-3}$',
    r'OT (ours)',
    r'$B$'
]

ncols = len(headers)

fig, axes = plt.subplots(
    N_ROWS,
    ncols,
    figsize=(14, 2.15*N_ROWS),
    dpi=180
)

if N_ROWS == 1:
    axes = axes[None, :]

for r in range(N_ROWS):

    for c in range(ncols):

        ax = axes[r,c]

        ax.imshow(
            rows[r][c]
        )

        ax.set_xticks([])
        ax.set_yticks([])

        # very subtle frame
        for spine in ax.spines.values():
            spine.set_linewidth(0.6)

        # column headings only once
        if r == 0:
            ax.set_title(
                headers[c],
                fontsize=11,
                pad=8
            )

    # pair number on left
    k = int(vis_pairs[r]["k"])

    axes[r,0].set_ylabel(
        f'pair {k}',
        fontsize=9,
        rotation=90,
        labelpad=8
    )


fig.suptitle(
    r'Gaussian $W_2$ covariance regularization vs. empirical optimal transport'
    '\n'
    r'All intermediate images use $\eta=0.5$ and the same CNT decoder/layout.',
    fontsize=13,
    y=0.998
)

plt.tight_layout(
    rect=[0,0,1,0.975],
    w_pad=0.35,
    h_pad=0.45
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
FIG_OUT = f'{OUT}/figs'
os.makedirs(FIG_OUT, exist_ok=True)

png_path = (
    f'{FIG_OUT}/gaussian_ridge_vs_ot.png'
)

pdf_path = (
    f'{FIG_OUT}/gaussian_ridge_vs_ot.pdf'
)

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches='tight'
)

plt.savefig(
    pdf_path,
    bbox_inches='tight'
)

plt.show()

print('\nSaved:')
print(png_path)
print(pdf_path)

In [ ]:
# ============================================================
# FINAL CORRECTED GAUSSIAN W2 vs OT — n=1000
#
# Uses:
#   covariance ridge = 1e-4
#   INNER matrix sqrt floor = 0.0   <-- corrected
#   same NSUB samples + NN extension as OT
#
# Creates NEW images/features, so previous broken Gaussian runs
# cannot be accidentally reused.
# ============================================================

import os, time, gc, json
import numpy as np
import torch
from PIL import Image
from scipy.stats import wilcoxon

N_TEST = min(1000, len(MAN_ALL))
ETA = 0.5
RIDGE = 1e-4

pairs = MAN_ALL[:N_TEST]
pair_ks = [int(p["k"]) for p in pairs]

# NEW names: don't reuse any previous Gaussian cache
GAUSS_FINAL_DIR = f'{SCALE}/gaussian_w2_final'
os.makedirs(GAUSS_FINAL_DIR, exist_ok=True)

GI_FINAL = f'{OUT}/feat_gaussian_w2_final_n{N_TEST}_inc.npy'
GD_FINAL = f'{OUT}/feat_gaussian_w2_final_n{N_TEST}_dino.npy'


# ------------------------------------------------------------
# Correct Gaussian W2 implementation
# ------------------------------------------------------------
@torch.no_grad()
def interp_gaussian_w2_final(d, eta=0.5, seed=0, ridge=RIDGE):

    L, Cc, H, W, sf, tf = _grids(d)
    N = sf.shape[0]

    # same deterministic sampling as OT
    i_s = _sub(N, seed, 's')
    i_t = _sub(N, seed, 't')

    X = sf[i_s].float()
    Y = tf[i_t].float()

    mu_x = X.mean(0)
    mu_y = Y.mean(0)

    Xc = X - mu_x
    Yc = Y - mu_y

    I = torch.eye(
        Cc,
        device=sf.device,
        dtype=torch.float32
    )

    Sa = _sym(
        Xc.T @ Xc / (len(X)-1)
    ) + ridge * I

    Sb = _sym(
        Yc.T @ Yc / (len(Y)-1)
    ) + ridge * I

    # source inverse sqrt remains stabilized
    Sa_half = _psd_sqrt(Sa, ridge)
    Sa_ihalf = _psd_invsqrt(Sa, ridge)

    middle = _sym(
        Sa_half @ Sb @ Sa_half
    )

    # IMPORTANT FIX:
    # no artificial eigenvalue floor in the middle sqrt
    middle_half = _psd_sqrt(
        middle,
        0.0
    )

    Amap = _sym(
        Sa_ihalf
        @ middle_half
        @ Sa_ihalf
    )

    # transport SAME sampled source cells
    transported_samples = (
        mu_y + Xc @ Amap.T
    )

    # identical NN-extension mechanism used by the other rules
    nn = torch.cdist(
        sf,
        sf[i_s]
    ).argmin(1)

    transported_full = transported_samples[nn]

    midpoint = (
        (1.0 - eta) * sf.float()
        + eta * transported_full
    )

    return _dec(
        d,
        midpoint,
        L, H, W, Cc
    )


# ============================================================
# 1. SANITY CHECK ON FIRST PAIR
# ============================================================

k = pair_ks[0]

A0 = Image.open(
    f'{SCALE}/A/{k:04d}.png'
).convert('RGB')

B0 = Image.open(
    f'{SCALE}/B/{k:04d}.png'
).convert('RGB')

d0 = build_d(
    cnt_load(A0),
    cnt_load(B0)
)

test_img = to_pil(
    interp_gaussian_w2_final(
        d0,
        ETA,
        k
    )
)

display(test_img)

# inspect affine-map spectrum
L,Cc,H,W,sf,tf = _grids(d0)

i_s = _sub(sf.shape[0], k, 's')
i_t = _sub(tf.shape[0], k, 't')

X = sf[i_s].float()
Y = tf[i_t].float()

mx,my = X.mean(0),Y.mean(0)

Xc = X-mx
Yc = Y-my

I = torch.eye(
    Cc,
    device=device
)

Sa = _sym(
    Xc.T@Xc/(len(X)-1)
) + RIDGE*I

Sb = _sym(
    Yc.T@Yc/(len(Y)-1)
) + RIDGE*I

Sh = _psd_sqrt(
    Sa,
    RIDGE
)

Sih = _psd_invsqrt(
    Sa,
    RIDGE
)

middle = _sym(
    Sh @ Sb @ Sh
)

Amap = _sym(
    Sih
    @ _psd_sqrt(middle,0.0)
    @ Sih
)

eig = torch.linalg.eigvalsh(Amap)

Ts = (
    my
    + Xc @ Amap.T
)

print(
    f"sanity: eig(A)="
    f"[{eig.min():.4f}, {eig.max():.2f}], "
    f"median |T(X)|={Ts.norm(dim=1).median():.3f}, "
    f"median |Y|={Y.norm(dim=1).median():.3f}"
)

del d0, test_img, X, Y, Ts
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 2. GENERATE CORRECTED GAUSSIAN IMAGES
#    resumable: existing final images are kept
# ============================================================

todo = [
    p for p in pairs
    if not os.path.exists(
        f'{GAUSS_FINAL_DIR}/{int(p["k"]):04d}.png'
    )
]

print(
    f'\nGaussian W2 final: '
    f'{N_TEST-len(todo)}/{N_TEST} cached; '
    f'{len(todo)} remaining.'
)

t0 = time.time()

for n,p in enumerate(todo,1):

    k = int(p["k"])

    A_img = Image.open(
        f'{SCALE}/A/{k:04d}.png'
    ).convert('RGB')

    B_img = Image.open(
        f'{SCALE}/B/{k:04d}.png'
    ).convert('RGB')

    d = build_d(
        cnt_load(A_img),
        cnt_load(B_img)
    )

    out = interp_gaussian_w2_final(
        d,
        eta=ETA,
        seed=k
    )

    to_pil(out).save(
        f'{GAUSS_FINAL_DIR}/{k:04d}.png'
    )

    del d,out

    if n % 50 == 0 or n == len(todo):

        elapsed = time.time()-t0
        rate = elapsed/n
        remain = (
            (len(todo)-n)
            * rate
            / 60
        )

        print(
            f'{n:4d}/{len(todo)} '
            f'{rate:.2f}s/pair  '
            f'ETA {remain:.1f} min'
        )

    if n % 100 == 0:
        gc.collect()
        torch.cuda.empty_cache()

if len(todo):
    print(
        f'Generation finished in '
        f'{(time.time()-t0)/60:.1f} min'
    )


# ============================================================
# 3. SCORE CORRECTED GAUSSIAN
# ============================================================

if (
    os.path.exists(GI_FINAL)
    and os.path.exists(GD_FINAL)
):

    GI = np.load(GI_FINAL)
    GD = np.load(GD_FINAL)

    print(
        '\nLoaded cached FINAL '
        'Gaussian features.'
    )

else:

    print(
        '\nScoring FINAL Gaussian outputs...'
    )

    ims = [
        Image.open(
            f'{GAUSS_FINAL_DIR}/{k:04d}.png'
        ).convert('RGB')
        for k in pair_ks
    ]

    ts = time.time()

    GI = inc_feats(
        ims,
        bs=64
    )

    GD = dino_feats(
        ims,
        bs=64
    )

    np.save(
        GI_FINAL,
        GI
    )

    np.save(
        GD_FINAL,
        GD
    )

    del ims

    print(
        f'Scoring finished in '
        f'{(time.time()-ts)/60:.1f} min'
    )


# ============================================================
# 4. LOAD CANONICAL OT FEATURES
# ============================================================

OI = np.load(
    f'{OUT}/feat_ot_inc.npy'
)[:N_TEST]

OD = np.load(
    f'{OUT}/feat_ot_dino.npy'
)[:N_TEST]


# ============================================================
# 5. REALISM
# ============================================================

rI_ot = realism(
    OI,
    MANI_INC
)

rI_g = realism(
    GI,
    MANI_INC
)

rD_ot = realism(
    OD,
    MANI_DINO
)

rD_g = realism(
    GD,
    MANI_DINO
)


# ============================================================
# 6. MIDPOINT PLACEMENT
# ============================================================

anc = dict(
    np.load(
        f'{OUT}/anchor_dino.npz'
    )
)

def placement_score(X):

    vals=[]

    for i,k in enumerate(pair_ks):

        a = anc[f'{k}_A']
        b = anc[f'{k}_B']

        da = np.linalg.norm(
            X[i]-a
        )

        db = np.linalg.norm(
            X[i]-b
        )

        vals.append(
            da/(da+db+1e-9)
        )

    return np.asarray(vals)


place_ot = placement_score(OD)
place_g  = placement_score(GD)

err_ot = np.abs(
    place_ot-0.5
)

err_g = np.abs(
    place_g-0.5
)


# ============================================================
# 7. FID + KID
# ============================================================

FID_ot = fid(
    OI,
    ref_inc
)

FID_g = fid(
    GI,
    ref_inc
)

KID_ot,KID_ot_sd = kid(
    OI,
    ref_inc,
    n_rep=50
)

KID_g,KID_g_sd = kid(
    GI,
    ref_inc,
    n_rep=50
)


# ============================================================
# 8. PAIRED STATISTICS
# ============================================================

def paired_test(
    name,
    g,
    o,
    higher_better=True
):

    d = (
        np.asarray(g)
        - np.asarray(o)
    )

    med = float(
        np.median(d)
    )

    lo,hi = boot_ci(
        d,
        n=5000,
        seed=17
    )

    try:
        p = float(
            wilcoxon(d).pvalue
        )
    except ValueError:
        p = 1.0

    if higher_better:
        gwr = np.mean(d>0)
    else:
        gwr = np.mean(d<0)

    print(
        f'{name:21s}'
        f'Δ(G-OT)={med:+.4f}  '
        f'95%CI [{lo:+.4f},{hi:+.4f}]  '
        f'p={p:.3g}  '
        f'Gaussian-win={100*gwr:.1f}%'
    )

    return {
        'delta_median': med,
        'ci_low': float(lo),
        'ci_high': float(hi),
        'p': p,
        'gaussian_win_rate': float(gwr)
    }


# ============================================================
# 9. FINAL RESULTS
# ============================================================

print(
    '\n'
    + '='*100
)

print(
    'FINAL CORRECTED Gaussian W2 vs OT '
    f'— n={N_TEST}, ridge={RIDGE}, '
    'inner sqrt floor=0'
)

print(
    '='*100
)

print(
    f'{"Method":14s}'
    f'{"FID↓":>10s}'
    f'{"KID↓":>18s}'
    f'{"RealInc↑":>12s}'
    f'{"RealDINO↑":>13s}'
    f'{"|place-.5|↓":>15s}'
)

print(
    f'{"OT":14s}'
    f'{FID_ot:10.1f}'
    f'{KID_ot:9.4f}±{KID_ot_sd:.4f}'
    f'{np.median(rI_ot):12.3f}'
    f'{np.median(rD_ot):13.3f}'
    f'{np.median(err_ot):15.3f}'
)

print(
    f'{"Gaussian W2":14s}'
    f'{FID_g:10.1f}'
    f'{KID_g:9.4f}±{KID_g_sd:.4f}'
    f'{np.median(rI_g):12.3f}'
    f'{np.median(rD_g):13.3f}'
    f'{np.median(err_g):15.3f}'
)

print(
    '\nPaired per-image comparisons:'
)

stats = {}

stats['inc'] = paired_test(
    'Inception realism',
    rI_g,
    rI_ot,
    True
)

stats['dino'] = paired_test(
    'DINO realism',
    rD_g,
    rD_ot,
    True
)

stats['placement'] = paired_test(
    'Placement error',
    err_g,
    err_ot,
    False
)

print(
    '\nSet-level differences:'
)

print(
    f'ΔFID  (Gaussian-OT) = '
    f'{FID_g-FID_ot:+.2f}'
)

print(
    f'ΔKID  (Gaussian-OT) = '
    f'{KID_g-KID_ot:+.5f}'
)


# ============================================================
# 10. SAVE FINAL NUMBERS
# ============================================================

RESULT = {
    'n': N_TEST,
    'ridge': RIDGE,
    'inner_sqrt_floor': 0.0,

    'OT': {
        'FID': FID_ot,
        'KID': KID_ot,
        'KID_std': KID_ot_sd,
        'RealInc': float(np.median(rI_ot)),
        'RealDINO': float(np.median(rD_ot)),
        'PlacementError': float(np.median(err_ot))
    },

    'GaussianW2': {
        'FID': FID_g,
        'KID': KID_g,
        'KID_std': KID_g_sd,
        'RealInc': float(np.median(rI_g)),
        'RealDINO': float(np.median(rD_g)),
        'PlacementError': float(np.median(err_g))
    },

    'paired': stats
}

SAVE_JSON = (
    f'{OUT}/'
    f'ot_vs_gaussian_w2_FINAL_n{N_TEST}.json'
)

with open(
    SAVE_JSON,
    'w'
) as f:
    json.dump(
        RESULT,
        f,
        indent=2
    )

print(
    f'\nSaved final results → '
    f'{SAVE_JSON}'
)